In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt
from collections import deque
import random

# Set default white background theme for matplotlib
plt.style.use('default')

# System parameters
K_WT = 1.0
K_PV = 1.0
K_T = 1.0
K_G = 1.0
K_PS = 120.0
B1, B2 = 0.425, 0.425
a12 = -1
T_WT = 1.5
T_PV = 1.3
T_T = 0.3
T_G = 0.08
T_PS = 20.0
R1, R2 = 2.4, 2.4
T12 = 0.545

# System dynamics
def power_system(state, t, load1, load2, P_PV_ref, P_WT_ref, K_P1, K_I1, K_D1, K_P2, K_I2, K_D2):
    dF1, P_g1, P_PV, dF2, P_g2, P_WT, P_tie, int_ACE1, int_ACE2 = state
    ACE1 = P_tie + B1 * dF1
    dACE1 = 2 * np.pi * T12 * (dF1 - dF2) + B1 * ((-dF1 / T_PS) + (K_PS / T_PS) * (P_g1 + P_PV - P_tie - load1))
    u1 = K_P1 * ACE1 + K_I1 * int_ACE1 + K_D1 * dACE1
    d_int_ACE1 = ACE1
    dP_g1 = (-P_g1 / T_G) + (K_G / T_G) * (-dF1 / R1 - u1)
    dP_PV = (-P_PV / T_PV) + (K_PV / T_PV) * P_PV_ref
    ddF1 = (-dF1 / T_PS) + (K_PS / T_PS) * (P_g1 + P_PV - P_tie - load1)
    ACE2 = -P_tie + B2 * dF2
    dACE2 = -2 * np.pi * T12 * (dF1 - dF2) + B2 * ((-dF2 / T_PS) + (K_PS / T_PS) * (P_g2 + P_WT + P_tie - load2))
    u2 = K_P2 * ACE2 + K_I2 * int_ACE2 + K_D2 * dACE2
    d_int_ACE2 = ACE2
    dP_g2 = (-P_g2 / T_G) + (K_G / T_G) * (-dF2 / R2 - u2)
    dP_WT = (-P_WT / T_WT) + (K_WT / T_WT) * P_WT_ref
    ddF2 = (-dF2 / T_PS) + (K_PS / T_PS) * (P_g2 + P_WT + P_tie - load2)
    dP_tie = 2 * np.pi * T12 * (dF1 - dF2)
    return [ddF1, dP_g1, dP_PV, ddF2, dP_g2, dP_WT, dP_tie, d_int_ACE1, d_int_ACE2]

# Cost function
def compute_advanced_itae(sol, t):
    dF1 = sol[:, 0]
    dF2 = sol[:, 3]
    dP_tie = sol[:, 6]
    error = np.abs(dF1) + np.abs(dF2) + np.abs(dP_tie)
    itae = np.trapezoid(t * error, t)
    settling_threshold = 0.02
    max_f1 = np.max(np.abs(dF1))
    max_f2 = np.max(np.abs(dF2))
    max_ptie = np.max(np.abs(dP_tie))
    f1_settled = np.where(np.abs(dF1) > settling_threshold * max_f1)[0]
    f2_settled = np.where(np.abs(dF2) > settling_threshold * max_f2)[0]
    ptie_settled = np.where(np.abs(dP_tie) > settling_threshold * max_ptie)[0]
    settling_time_f1 = t[f1_settled[-1]] if f1_settled.size > 0 else 0
    settling_time_f2 = t[f2_settled[-1]] if f2_settled.size > 0 else 0
    settling_time_ptie = t[ptie_settled[-1]] if ptie_settled.size > 0 else 0
    overshoot_penalty = (np.max(dF1)**2 + np.max(dF2)**2 + np.max(dP_tie)**2) * 100
    undershoot_penalty = (np.abs(np.min(dF1))**2 + np.abs(np.min(dF2))**2 + np.abs(np.min(dP_tie))**2) * 100
    settling_penalty = (settling_time_f1 + settling_time_f2 + settling_time_ptie) * 10
    total_cost = itae + overshoot_penalty + undershoot_penalty + settling_penalty
    return total_cost

# Neural Network
class ImprovedDQN:
    def __init__(self, state_dim, action_dim, hidden_dims=[64, 32]):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_dims = hidden_dims
        layers = [state_dim] + hidden_dims + [action_dim]
        self.weights = []
        self.biases = []
        for i in range(len(layers) - 1):
            w = np.random.randn(layers[i], layers[i+1]) * np.sqrt(2.0 / layers[i])
            b = np.zeros(layers[i+1])
            self.weights.append(w)
            self.biases.append(b)
    def forward(self, state):
        x = state
        for i in range(len(self.weights) - 1):
            x = np.maximum(0, np.dot(x, self.weights[i]) + self.biases[i])
        x = np.dot(x, self.weights[-1]) + self.biases[-1]
        return x
    def update(self, state, action, target, lr=0.0001):
        activations = [state]
        for i in range(len(self.weights) - 1):
            z = np.dot(activations[-1], self.weights[i]) + self.biases[i]
            a = np.maximum(0, z)
            activations.append(a)
        z_out = np.dot(activations[-1], self.weights[-1]) + self.biases[-1]
        activations.append(z_out)
        error = target - activations[-1][action]
        error = np.clip(error, -1e3, 1e3)
        grad_w_out = np.zeros_like(self.weights[-1])
        grad_w_out[:, action] = activations[-2] * error
        grad_b_out = np.zeros_like(self.biases[-1])
        grad_b_out[action] = error
        self.weights[-1] += lr * grad_w_out
        self.biases[-1] += lr * grad_b_out
        delta = np.zeros_like(activations[-2])
        delta = self.weights[-1][:, action] * error
        for i in range(len(self.weights) - 2, -1, -1):
            delta[activations[i+1] <= 0] = 0
            grad_w = np.outer(activations[i], delta)
            grad_b = delta
            self.weights[i] += lr * grad_w
            self.biases[i] += lr * grad_b
            if i > 0:
                delta = np.dot(delta, self.weights[i].T)

# Replay Buffer
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.array, zip(*batch))
        return state, action, reward, next_state, done
    def __len__(self):
        return len(self.buffer)

# Performance Metrics
def compute_metrics(signal, t, steady_state=0, settling_threshold=0.02):
    overshoot = np.max(signal) if np.max(signal) > steady_state else 0
    undershoot = -np.min(signal) if np.min(signal) < steady_state else 0
    max_amplitude = np.max(np.abs(signal))
    threshold = settling_threshold * max_amplitude
    settled = np.where(np.abs(signal) > threshold)[0]
    settling_time = t[settled[-1]] if settled.size > 0 else t[-1]
    return settling_time, overshoot, undershoot

# Advanced DQN Optimization with Robust Load Handling
def advanced_dqn_optimize(objective_func, bounds, episodes=400, gamma=0.95,
                         epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.997,
                         batch_size=32, target_update=5):
    state_dim = 15
    action_dim = len(bounds) * 5
    main_net = ImprovedDQN(state_dim, action_dim)
    target_net = ImprovedDQN(state_dim, action_dim)
    for i in range(len(main_net.weights)):
        target_net.weights[i] = main_net.weights[i].copy()
        target_net.biases[i] = main_net.biases[i].copy()
    replay_buffer = ReplayBuffer()
    epsilon = epsilon_start
    gains = np.array([(b[0] + b[1]) / 2 for b in bounds])
    best_gains = gains.copy()
    best_score = float('inf')
    t = np.linspace(0, 20, 2000)  # Extended to 20 seconds
    load2 = 0.0
    P_PV_ref, P_WT_ref = 0.08, 0.06
    state0 = [0, 0, 0, 0, 0, 0, 0, 0, 0]
    load_variations = [0.10, 0.15, 0.20, 0.25]  # 10%, 15%, 20%, 25%

    for episode in range(episodes):
        if episode > 0 and episode % 50 == 0:
            gains = best_gains + np.random.normal(0, 0.1, len(gains))
            gains = np.clip(gains, [b[0] for b in bounds], [b[1] for b in bounds])
        episode_reward = 0
        steps_per_episode = 30
        for step in range(steps_per_episode):
            # Average cost over multiple load conditions
            total_cost = 0
            for load_var in load_variations:
                load1 = 0.1 * (1 + load_var)
                sol = odeint(power_system, state0, t[:200],
                            args=(load1, load2, P_PV_ref, P_WT_ref,
                                  gains[0], gains[1], gains[2],
                                  gains[0], gains[1], gains[2]))
                current_gains = gains  # Use the gains directly here
                cost = objective_func(current_gains)
                total_cost += cost / len(load_variations)
            score = total_cost

            sol = odeint(power_system, state0, t[:200],
                        args=(0.1, load2, P_PV_ref, P_WT_ref,
                              gains[0], gains[1], gains[2],
                              gains[0], gains[1], gains[2]))
            dF1 = sol[:, 0]
            dF2 = sol[:, 3]
            dP_tie = sol[:, 6]
            dF1_dt = np.gradient(dF1, t[:200])[-1]
            dF2_dt = np.gradient(dF2, t[:200])[-1]
            dP_tie_dt = np.gradient(dP_tie, t[:200])[-1]
            state = np.concatenate([sol[-1], gains, [dF1_dt, dF2_dt, dP_tie_dt]])

            if np.random.rand() < epsilon:
                action = np.random.randint(action_dim)
            else:
                q_values = main_net.forward(state)
                action = np.argmax(q_values)

            param_idx = action // 5
            action_type = action % 5
            adjustments = [-0.1, -0.025, 0.0, 0.025, 0.1]
            new_gains = gains.copy()
            if param_idx < len(gains):
                new_gains[param_idx] += adjustments[action_type]
                new_gains[param_idx] = np.clip(new_gains[param_idx], bounds[param_idx][0], bounds[param_idx][1])

            total_cost_new = 0
            for load_var in load_variations:
                load1 = 0.1 * (1 + load_var)
                sol_new = odeint(power_system, state0, t[:200],
                                args=(load1, load2, P_PV_ref, P_WT_ref,
                                      new_gains[0], new_gains[1], new_gains[2],
                                      new_gains[0], new_gains[1], new_gains[2]))
                current_new_gains = new_gains  # Use the new gains directly here
                cost_new = objective_func(current_new_gains)
                total_cost_new += cost_new / len(load_variations)
            score_new = total_cost_new

            improvement = score - score_new
            sol_new_base = odeint(power_system, state0, t[:200],
                                 args=(0.1, load2, P_PV_ref, P_WT_ref,
                                       new_gains[0], new_gains[1], new_gains[2],
                                       new_gains[0], new_gains[1], new_gains[2]))
            f1_settling, f1_overshoot, _ = compute_metrics(sol_new_base[:, 0], t[:200])
            f2_settling, f2_overshoot, _ = compute_metrics(sol_new_base[:, 3], t[:200])
            ptie_settling, ptie_overshoot, _ = compute_metrics(sol_new_base[:, 6], t[:200])
            reward = (improvement * 100) - (30 * f1_settling) - (30 * f2_settling) - (30 * ptie_settling) - (500 * f1_overshoot) - (500 * f2_overshoot) - (1000 * ptie_overshoot)
            if score_new < 1.0:
                reward += 50
            if score_new > 5.0:
                reward -= 100

            dF1_new = sol_new_base[:, 0]
            dF2_new = sol_new_base[:, 3]
            dP_tie_new = sol_new_base[:, 6]
            dF1_dt_new = np.gradient(dF1_new, t[:200])[-1]
            dF2_dt_new = np.gradient(dF2_new, t[:200])[-1]
            dP_tie_dt_new = np.gradient(dP_tie_new, t[:200])[-1]
            next_state = np.concatenate([sol_new_base[-1], new_gains, [dF1_dt_new, dF2_dt_new, dP_tie_dt_new]])
            done = (step == steps_per_episode - 1)
            replay_buffer.push(state, action, reward, next_state, done)
            gains = new_gains
            episode_reward += reward
            if score_new < best_score:
                best_score = score_new
                best_gains = new_gains.copy()
            if len(replay_buffer) > batch_size:
                states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)
                for i in range(batch_size):
                    target = rewards[i]
                    if not dones[i]:
                        next_q_values = target_net.forward(next_states[i])
                        target += gamma * np.max(next_q_values)
                    main_net.update(states[i], actions[i], target, lr=0.0001)
        if episode % target_update == 0:
            for i in range(len(main_net.weights)):
                target_net.weights[i] = main_net.weights[i].copy()
                target_net.biases[i] = main_net.biases[i].copy()
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        if episode % 20 == 0:
            print(f"Episode {episode}: Best Score = {best_score:.4f}, Epsilon = {epsilon:.3f}")
            print(f"Best Gains: K_P = {best_gains[0]:.3f}, K_I = {best_gains[1]:.3f}, K_D = {best_gains[2]:.3f}")
    return best_gains, best_score

# Main Execution
if __name__ == "__main__":
    t = np.linspace(0, 20, 2000)
    load2 = 0.0
    P_PV_ref = 0.08
    P_WT_ref = 0.06
    state0 = [0, 0, 0, 0, 0, 0, 0, 0, 0]

    def pid_objective(gains):
        K_P, K_I, K_D = gains  # Unpack only the gains (3 values)
        sol = odeint(power_system, state0, t, args=(0.1, load2, P_PV_ref, P_WT_ref, K_P, K_I, K_D, K_P, K_I, K_D))
        return compute_advanced_itae(sol, t)

    pid_bounds = [(0.1, 5.0), (0.1, 2.0), (0.01, 1.0)]
    print("Starting Advanced DQN optimization with robust load handling...")
    pid_best_gains_dqn, pid_best_score_dqn = advanced_dqn_optimize(pid_objective, pid_bounds, episodes=400)
    K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN = pid_best_gains_dqn

    # Simulate with optimized gains for different load variations
    load_variations = [0.10, 0.15, 0.20, 0.25]
    colors = ['#1f4e79', '#8B0000', '#228B22', '#FF4500']

    plt.figure(figsize=(15, 10))

    # Area 1 Frequency Deviation (ΔF1)
    plt.subplot(3, 1, 1)
    for i, load_var in enumerate(load_variations):
        load1 = 0.1 * (1 + load_var)
        sol = odeint(power_system, state0, t, args=(load1, load2, P_PV_ref, P_WT_ref,
                                                   K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN,
                                                   K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN))
        plt.plot(t, sol[:, 0], label=f"{int(load_var * 100)}% Load", color=colors[i], linewidth=2)
    plt.ylabel("ΔF1 (Hz)", fontsize=11)
    plt.title("System Response for Different Load Variations (Robust DQN-PID)", fontsize=16, fontweight='bold', pad=20)
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)

    # Area 2 Frequency Deviation (ΔF2)
    plt.subplot(3, 1, 2)
    for i, load_var in enumerate(load_variations):
        load1 = 0.1 * (1 + load_var)
        sol = odeint(power_system, state0, t, args=(load1, load2, P_PV_ref, P_WT_ref,
                                                   K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN,
                                                   K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN))
        plt.plot(t, sol[:, 3], label=f"{int(load_var * 100)}% Load", color=colors[i], linewidth=2)
    plt.ylabel("ΔF2 (Hz)", fontsize=11)
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)

    # Tie-Line Power Deviation (ΔP_tie)
    plt.subplot(3, 1, 3)
    for i, load_var in enumerate(load_variations):
        load1 = 0.1 * (1 + load_var)
        sol = odeint(power_system, state0, t, args=(load1, load2, P_PV_ref, P_WT_ref,
                                                   K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN,
                                                   K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN))
        plt.plot(t, sol[:, 6], label=f"{int(load_var * 100)}% Load", color=colors[i], linewidth=2)
    plt.xlabel("Time (s)", fontsize=12)
    plt.ylabel("ΔP_tie (p.u.)", fontsize=11)
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Compute and print performance metrics
    print("\n" + "="*70)
    print("PERFORMANCE METRICS FOR DIFFERENT LOAD VARIATIONS")
    print("="*70)
    print(f"{'Load Variation (%)':<15} {'Signal':<8} {'Settling Time (s)':<18} {'Peak Overshoot':<15} {'Peak Undershoot':<15}")
    print("-"*70)
    for load_var in load_variations:
        load1 = 0.1 * (1 + load_var)
        sol = odeint(power_system, state0, t, args=(load1, load2, P_PV_ref, P_WT_ref,
                                                  K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN,
                                                  K_P1_PID_DQN, K_I1_PID_DQN, K_D1_PID_DQN))
        st_f1, os_f1, us_f1 = compute_metrics(sol[:, 0], t)
        st_f2, os_f2, us_f2 = compute_metrics(sol[:, 3], t)
        st_tie, os_tie, us_tie = compute_metrics(sol[:, 6], t)
        print(f"{int(load_var * 100):<15} {'ΔF1':<8} {st_f1:<18.3f} {os_f1:<15.6f} {us_f1:<15.6f}")
        print(f"{int(load_var * 100):<15} {'ΔF2':<8} {st_f2:<18.3f} {os_f2:<15.6f} {us_f2:<15.6f}")
        print(f"{int(load_var * 100):<15} {'ΔP_tie':<8} {st_tie:<18.3f} {os_tie:<15.6f} {us_tie:<15.6f}")